In [ ]:
#--------Privacy-Preserving Distributed Graph Learning/federated learning--- Multiple hospitals train a shared model while keeping their local patient records at their own sites.
#- It reduces the need to centralize raw records, but it does not automatically guarantee privacy.
"""
Imagine three hospitals have their own de-identified research datasets. Each hospital trains a local model to classify 
whether a documented follow-up outcome is present. They share model updates rather than patient-level records.

    Each hospital trained a copy of the shared model using its own local data.
    The hospitals' updated model parameters were averaged.
    The averaged parameters became the starting point for the next training round.

Important privacy limitations: This simulation does not implement secure aggregation, differential privacy, authentication, 
or protection against model-update leakage. Real federated graph learning also requires careful handling of relationships 
that cross hospital boundaries. Federated learning alone is not a privacy guarantee.

        Block 1 — Three hospitals have local data
        Each hospital has its own synthetic training examples

        Block 2 — Create a shared model
        Start with the same initial model parameters

        Block 3 — Train locally
        Each hospital trains its own copy using its own data

        Block 4 — Share model parameters
        The simulated coordinator receives updated model weights and biases, not the training rows

        Block 5 — Average the parameters
        Combine the three hospitals' trained model parameters

        Block 6 — Repeat and use the shared model
        Send the averaged model back to hospitals for the next round

Unlike the earlier examples, this model is not navigating graph nodes or scoring graph edges. It is a simple binary classifier 
used to demonstrate federated training.
Step	What happens
----    ------------
1	    The hospital's model predicts whether each outcome is documented.
2	    The loss function measures prediction errors.
3	    backward() calculates gradients showing how the parameters affect the error.
4	    step() updates the model weights and bias.
5	    The process repeats until 20 local updates have been performed.
"""
import torch
from torch import nn
from copy import deepcopy
torch.manual_seed(42)
#------------Block 1 — Understand the local hospital data
hospital_data = [                                          # Each hospital has a pair of tensors: (X, y). X = input features used for training. y = correct labels the model is supposed to predict.
    (
        torch.tensor([[1., 0.],                            # Hosppital 1 - This is x data - feature 1 and feature 2
                      [1., 1.],                            # This is x data - feature 1 and feature 2
                      [0., 0.]]),                          # This is x data - feature 1 and feature 2
        torch.tensor([[1.], [1.], [0.]])                   # This is Label (Y) data 
    ),
"""
Training example	Feature 1	Feature 2	Label (y)
1	                    1	        0	       1
2	                    1	        1	       1    
3	                    0	        0	       0
Feature 1 represents whether biomarker information is present and Feature 2 represents whether treatment information 
is present. A label of 1 means a follow-up outcome is documented, while 0 means it is not.
"""
    (
        torch.tensor([[0., 1.],                           #-- This is hospital 2 block  (X_hospital_2, y_hospital_2),
                      [1., 1.], 
                      [0., 0.]]),
        torch.tensor([[0.], [1.], [0.]])
    ),
    (
        torch.tensor([[1., 0.],                           #-- This is hospital 3 block  (X_hospital_3, y_hospital_3),
                      [0., 1.], 
                      [1., 1.]]),
        torch.tensor([[1.], [0.], [1.]])
    )
]
#-----------Block 2 — Create the shared model
def create_model():
    return nn.Linear(2, 1)
global_model = create_model()

"""
This creates a simple neural model with: - 2 input features and 1 output score
The mathematical structure is: [z = w_1x_1+w_2x_2+ b] Here:
- (x_1,x_2) are the input features.
- (w_1,w_2) are trainable weights.
- (b) is a trainable bias.
- (z) is the raw output score, also called a logit.
A sigmoid function converts the logit into a value between 0 and 1, interpreted here as the model's estimated probability that 
the outcome is documented.
"""
#------End of Block 2 — Create the shared model

#----Block 3 — Each hospital trains its own model
loss_function = nn.BCEWithLogitsLoss()
for round_number in range(10):                        #--range(10) means the hospitals will participate in 10 rounds of collaborative training.  Repeat the process and test the shared model
    local_models = []                                 # local_models = [] creates an empty list to store the three trained local models for the current round.
    # Each hospital starts from the shared model.
    for X_local, y_local in hospital_data:
        local_model = deepcopy(global_model)          # creates a separate copy of the current shared model for each hospital. 
        optimizer = torch.optim.SGD(                  # SGD - Stochastic Gradient Descent., because each iteration uses the entire local dataset, the update is effectively a full-batch gradient descent step.
            local_model.parameters(), lr=0.2
        )
        # Train locally; raw data stays at this simulated hospital.
        for _ in range(20):
            logits = local_model(X_local)
            loss = loss_function(logits, y_local)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        local_models.append(local_model)
#--- Block 5 — Average the three hospital models. Each model has three trainable parameters: Weight 1, Weight 2 and Bias.
"""
Suppose the local models end up with these illustrative values:
Parameter	Hospital 1	Hospital 2	Hospital 3	    Average
Weight 1	    0.8	        0.5	        1.1	        0.8
Weight 2	    0.4	        0.7	        0.1     	0.4
Bias	        -0.2	    0.1	        0.4	        0.1
For Weight 1, the calculation is: [frac{0.8+0.5+1.1}{3}=0.8]

The same averaging operation is performed independently for Weight 2 and Bias.
The averaged parameters replace the parameters in global_model.
"""
with torch.no_grad():
        for parameter_index, global_parameter in enumerate(
            global_model.parameters()                            # Access the shared model's trainable parameters.
        ):
            global_parameter.copy_(                              # Replace the shared parameter with the calculated average.
                torch.stack([                                    # Stack the three corresponding parameter tensors together.
                    list(model.parameters())[parameter_index]    # Retrieve the corresponding parameter from each hospital's model
                    for model in local_models
                ]).mean(dim=0)                                   # Calculate their element-wise average.
            )
print("Shared model trained across 3 simulated hospitals.")
with torch.no_grad():                                            # prevents gradient tracking during this parameter replacement.
    example = torch.tensor([[1., 1.]])
    probability = torch.sigmoid(global_model(example)).item()             #-- A sigmoid function

print("Example model score:", round(probability, 3))      

"""
For a real deployment, each hospital would send its model updates to a coordinator over a secured communication channel. 
Your code simulates this collaboration within a single Python process; it does not implement actual hospital-to-hospital 
networking. Also, the code averages full model parameters, not just the changes in parameters from the initial model. 
Averaging the full parameters is the standard FedAvg approach when all local models start from the same global model.
"""

Shared model trained across 3 simulated hospitals.
Example model score: 0.913
